# Large Language Models

Modelos de linguagem de larga escala (LLMs) são modelos probabilísticos treinados para estimar a distribuição de sequências de texto. Em sua forma mais comum, são modelos autoregressivos que fatoram a probabilidade conjunta como

$$
P(x_1, x_2, \dots, x_T) = \prod_{t=1}^{T} P(x_t \mid x_1, \dots, x_{t-1})
$$

e aprendem a prever o próximo token dado o contexto anterior. Na prática, esses modelos são implementados com arquiteturas baseadas em Transformers, que permitem capturar dependências de longo alcance e escalar para grandes volumes de dados e parâmetros. Como consequência, LLMs são capazes de realizar diversas tarefas com o mesmo modelo, incluindo geração de texto, tradução, resumo e resposta a perguntas.

## Hugging Face

Nesta seção, apresentamos o fluxo essencial de uso de um modelo de linguagem autoregressivo com a biblioteca `transformers`: carregar tokenizador e modelo, transformar texto em tokens, interpretar os IDs gerados, reconstruir o texto por decodificação e gerar novas sequências. Utilizaremos o **GPT-2**, que é leve e suficiente para demonstrar os conceitos principais.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

### Carregando modelo e tokenizador

O tokenizador converte texto em sequências de inteiros (**token IDs**), que são a entrada do modelo. O modelo, por sua vez, produz distribuições de probabilidade sobre o próximo token.

In [ ]:
model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print("Dispositivo:", device)
print("Vocab size:", tokenizer.vocab_size)

### Carregamento, tokenização e tokens especiais

O tokenizador converte texto em **token IDs**, que são a entrada do modelo. Além dos tokens “normais”, existem **tokens especiais** que controlam o comportamento do modelo, como `eos_token` (fim de sequência) e, em alguns modelos, `pad_token`, `bos_token`, entre outros. O GPT-2 não define `pad_token` por padrão, então é comum reutilizar o `eos_token` para viabilizar processamento em batch.

A tokenização retorna principalmente `input_ids` e `attention_mask`. A máscara indica quais posições correspondem a tokens reais (1) e quais são apenas preenchimento (0). Também podemos converter IDs de volta para tokens e reconstruir o texto com decodificação.

In [ ]:
text = "Deep learning models can generate text."
encoded = tokenizer(text, return_tensors="pt")

input_ids = encoded["input_ids"]
attention_mask = encoded["attention_mask"]

print("input_ids:", input_ids)
print("attention_mask:", attention_mask)

tokens = tokenizer.convert_ids_to_tokens(input_ids[0])
print("tokens:", tokens)

In [ ]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.eos_token_id

print("pad_token:", tokenizer.pad_token)
print("pad_token_id:", tokenizer.pad_token_id)
print("eos_token:", tokenizer.eos_token)
print("eos_token_id:", tokenizer.eos_token_id)

### Decodificação

A decodificação reconstrói o texto a partir dos IDs. Como a tokenização pode quebrar palavras, essa etapa é importante para obter uma sequência legível.

In [ ]:
decoded = tokenizer.decode(input_ids[0])
print("original:", text)
print("decoded:", decoded)

### Tokenização em batch

Ao trabalhar com múltiplas entradas, utilizamos padding para igualar os comprimentos. No GPT-2, é comum usar o token de fim de sequência como padding.

In [ ]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.eos_token_id

texts = [
    "Language models predict the next token.",
    "Transformers are powerful sequence models.",
    "Tokenization is the first step in the pipeline."
]

batch = tokenizer(texts, return_tensors="pt", padding=True, truncation=True)

print("input_ids shape:", batch["input_ids"].shape)
print("attention_mask shape:", batch["attention_mask"].shape)

### Passagem pelo modelo

Uma vez que o texto é convertido em tokens, podemos fornecê-los diretamente ao modelo. Internamente, o modelo processa a sequência completa considerando o contexto de cada posição e produz, para cada token de entrada, um vetor de valores associado a todo o vocabulário.

Esses valores são chamados de **logits** e representam scores não normalizados para cada possível próximo token. Para uma sequência de comprimento $T$ e um vocabulário de tamanho $V$, a saída do modelo possui dimensão $(B, T, V)$, onde $B$ é o tamanho do batch. Cada posição da sequência contém, portanto, uma distribuição potencial sobre o próximo token, que pode ser transformada em probabilidades por meio de uma função softmax.

In [ ]:
text = "The capital of France is"
encoded = tokenizer(text, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model(**encoded)

print("logits shape:", outputs.logits.shape)

In [ ]:
last_logits = outputs.logits[:, -1, :] # A última posição da sequência contém a predição do próximo token
next_id = last_logits.argmax(dim=-1).item()
next_token = tokenizer.decode([next_id])

print("entrada:", text)
print("próximo token:", repr(next_token))

In [ ]:
k = 5

probs = torch.softmax(last_logits, dim=-1)
top_probs, top_ids = torch.topk(probs, k=k) # Podemos observar os k tokens mais prováveis para a próxima posição

for i, (tid, p) in enumerate(zip(top_ids[0], top_probs[0]), 1):
    print(i, tokenizer.decode([tid.item()]), f"{p.item():.4f}")

### Geração de texto

A geração autoregressiva consiste em produzir novos tokens de forma iterativa: dado um prompt inicial, o modelo prevê uma distribuição de probabilidade para o próximo token, seleciona um deles e o adiciona à sequência. Esse processo é repetido até atingir um critério de parada, como um número máximo de tokens ou a geração de um token especial de fim de sequência.

Na biblioteca `transformers`, esse processo é encapsulado pelo método `generate`, que implementa diferentes estratégias de decodificação. No modo **greedy** (`do_sample=False`), o modelo seleciona sempre o token de maior probabilidade em cada passo, resultando em saídas determinísticas. Alternativamente, ao ativar `do_sample=True`, o próximo token é amostrado da distribuição prevista, permitindo maior diversidade. Parâmetros como `temperature` controlam a suavização da distribuição (valores menores tornam a saída mais determinística), enquanto `top_k` limita a escolha aos $k$ tokens mais prováveis em cada passo.

Assim, o método `generate` abstrai o loop autoregressivo e permite controlar diretamente o comportamento da geração por meio de seus parâmetros.

In [ ]:
prompt = "The future of artificial intelligence is"

inputs = tokenizer(prompt, return_tensors="pt").to(device)

output_ids = model.generate(
    **inputs,
    max_new_tokens=30,
    do_sample=False  # greedy decoding
)

generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)

print("Prompt:")
print(prompt)

print("\nTexto gerado:")
print(generated_text)

Também podemos ativar amostragem estocástica, permitindo maior diversidade nas saídas. Nesse caso, em vez de sempre escolher o token mais provável, o modelo amostra a partir da distribuição de probabilidade.

In [ ]:
output_ids = model.generate(
    **inputs,
    max_new_tokens=30,
    do_sample=True,
    temperature=0.8,
    top_k=50
)

generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)

print("Texto gerado (amostragem):")
print(generated_text)

## Fine-tuning eficiente com LoRA

Em modelos de linguagem, o fine-tuning completo costuma ser custoso, pois envolve a atualização de todos os parâmetros da rede. Uma alternativa mais eficiente é o **LoRA** (*Low-Rank Adaptation*), que mantém os pesos originais do modelo congelados e introduz pequenas matrizes treináveis em módulos específicos da arquitetura.

A principal ideia é simples: em vez de alterar diretamente uma matriz de pesos grande $W$, aprendemos uma atualização de baixa dimensão

$$
W' = W + \Delta W, \quad \Delta W = BA
$$

onde $A$ e $B$ são matrizes de posto reduzido. Assim, o número de parâmetros treináveis diminui significativamente, reduzindo custo computacional e consumo de memória.

Nesta seção, utilizamos um modelo moderno instruction-tuned e um dataset sintético simples, com o objetivo de mostrar como um LLM pode aprender uma transformação estruturada a partir de exemplos.

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset
import torch

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### Carregando modelo e tokenizador

O modelo utilizado é o **Qwen2.5-0.5B-Instruct**, que já foi ajustado para seguir instruções no formato de chat.

Por esse motivo, as entradas devem ser estruturadas como uma sequência de mensagens com papéis definidos (`system`, `user`, `assistant`). O tokenizador fornece um *chat template* que converte essa estrutura em texto adequado para o modelo.a

In [ ]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
base_model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

print("Dispositivo:", device)
print("Modelo carregado:", model_name)

### Construindo um dataset sintético

Em vez de utilizar linguagem natural tradicional, o dataset foi construído para representar uma tarefa estruturada. Cada exemplo consiste em uma entrada no formato nome e idade, por exemplo "John is 25", e uma saída correspondente no formato simbólico "<J#25#625>". Nesse padrão:  
- A primeira posição representa a inicial do nome  
- A segunda posição representa a idade  
- A terceira posição representa o quadrado da idade  

O objetivo do modelo é aprender essa transformação a partir de exemplos, funcionando como uma forma simples de aprendizado de mapeamento simbólico.

In [ ]:
import random

names = ["John", "Mary", "Alice", "Bob", "Carol", "David", "Zoe", "Lucas", "Ana", "Pedro"]

examples = {"input": [], "output": []}

for _ in range(100):
    name = random.choice(names)
    age = random.randint(1, 50)

    examples["input"].append(f"{name} is {age}")
    examples["output"].append(f"<{name[0]}#{age}#{age**2}>")

dataset = Dataset.from_dict(examples)
dataset

### Formatação com chat template

Cada exemplo é convertido para o formato esperado pelo modelo, incluindo uma instrução global no papel `system`, a entrada no papel `user` e a saída esperada no papel `assistant`. O uso do `system` define explicitamente a tarefa que o modelo deve executar ("Extract information"), enquanto o `chat template` garante que a sequência final esteja alinhada com o formato utilizado no treinamento original do modelo.

In [ ]:
def format_example(example):
    messages = [
        {"role": "system", "content": "Extract information."},
        {"role": "user", "content": example["input"]},
        {"role": "assistant", "content": example["output"]}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    return {"text": text}

dataset = dataset.map(format_example)
dataset[0]["text"]

### Tokenização

Após a formatação textual, os exemplos são tokenizados. Como se trata de um modelo autoregressivo, utilizamos os próprios `input_ids` como rótulos, fazendo com que o modelo aprenda a prever cada próximo token da sequência, incluindo a resposta do assistente. O truncamento e o padding são aplicados para manter o tamanho das sequências fixo.

In [ ]:
max_length = 64

def tokenize_function(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=max_length
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_dataset = dataset.map(tokenize_function, batched=False)
tokenized_dataset = tokenized_dataset.remove_columns(["input", "output", "text"])

tokenized_dataset[0]

### Configuração do LoRA

Os adaptadores LoRA são inseridos em módulos de projeção da arquitetura transformer (`q_proj`, `k_proj`, `v_proj`, `o_proj`), que são responsáveis pelas transformações lineares no mecanismo de atenção e, portanto, são pontos eficazes para adaptação. A partir desse momento:  
- O modelo base permanece congelado  
- Apenas os parâmetros introduzidos pelo LoRA são treinados  

In [ ]:
# for name, module in model.named_modules():
#     if "proj" in name:
#         print(name)

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

### Treinamento

O treinamento é realizado apenas sobre os parâmetros adicionados pelo LoRA. Como o dataset é pequeno e sintético, poucas épocas são suficientes para que o modelo capture o padrão da transformação. O objetivo aqui não é generalização ampla, mas demonstrar a capacidade de adaptação rápida do modelo a uma tarefa específica.

In [ ]:
training_args = TrainingArguments(
    output_dir="./qwen_lora_toy",
    per_device_train_batch_size=2,
    num_train_epochs=5,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="no",
    report_to="none"
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

trainer.train()

### Comparação entre modelo base e modelo com LoRA

Para avaliar o efeito do fine-tuning, comparamos:  
- O modelo base (sem adaptação)  
- O modelo com LoRA  

O modelo base tende a produzir respostas genéricas ou desalinhadas com a tarefa, pois não foi treinado para esse tipo específico de transformação. Já o modelo com LoRA aprende o padrão presente no dataset e passa a gerar saídas estruturadas corretamente, evidenciando que pequenas atualizações direcionadas são suficientes para adaptar um LLM a tarefas bem definidas.

In [ ]:
def generate(model, messages, max_new_tokens=20):
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    input_length = inputs["input_ids"].shape[1]
    new_tokens = output_ids[0][input_length:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True)

    generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return response

In [ ]:
messages = [
    {"role": "system", "content": "Extract information."},
    {"role": "user", "content": "Silvan's age is 8"}
]

lora_generated_text = generate(model, messages)

print("Modelo com LoRA:")
print(lora_generated_text)

In [ ]:
messages = [
    {"role": "system", "content": "Extract information."},
    {"role": "user", "content": "Silvan's age is 8"}
]

base_generated_text = generate(base_model, messages)
lora_generated_text = generate(model, messages)

print("Modelo base:")
print(base_generated_text)
print()
print("Modelo com LoRA:")
print(lora_generated_text)

In [ ]:
lora_generated_text = generate(model, messages)

print("Modelo com LoRA:")
print(lora_generated_text)

## Exercícios

### Exercício 1

Implemente um processo completo de fine-tuning utilizando LoRA em um modelo de linguagem, seguindo a mesma estrutura apresentada neste notebook. Escolha um dataset próprio, que pode ser construído manualmente ou obtido de fontes externas, desde que represente uma tarefa clara de transformação ou geração de texto.

O dataset deve conter pares de entrada e saída coerentes com uma tarefa bem definida, como reformulação de frases, classificação textual convertida em formato gerativo, extração de informações ou tradução simples. A estrutura dos dados deve ser adaptada para o formato de chat (`system`, `user`, `assistant`), conforme utilizado no exemplo.

Realize as etapas de formatação, tokenização e treinamento com LoRA, e avalie qualitativamente os resultados gerados pelo modelo após o fine-tuning. Compare o comportamento do modelo base com o modelo ajustado, discutindo se o modelo aprendeu o padrão desejado, se houve memorização ou generalização, e quais limitações foram observadas.